[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leaguilar/fit_2026/blob/main/notebooks/1_redes_neuronales_backpropagation.ipynb)

# 1. Redes neuronales y backpropagation

**Taller Hands-On AI | FIT 2026**

En el notebook anterior movimos **dos** perillas. Una red neuronal tiene
muchas más, encadenadas en capas. Falta una sola pieza: cómo calcular el
gradiente cuando todo está encadenado.

Esa pieza es la **regla de la cadena**, y aplicada a una red se llama
**backpropagation**.

| Parte | Qué hacemos |
|---|---|
| 1 | Una neurona: qué forma tiene. |
| 2 | Sumar neuronas para construir una curva, **a mano**. |
| 3 | La regla de la cadena, escrita y comprobada. |
| 4 | La red completa en 25 líneas de numpy. |
| 5 | La red es un **interpolador**: interpola bien, extrapola mal. |
| 6 | Lo mismo en PyTorch, en 10 líneas. |

In [ ]:
# --- Setup ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9

# Los tonos estan elegidos para que se distingan tambien con daltonismo
# rojo-verde (comprobado por script, no a ojo).
AZUL   = "#215CAF"   # el modelo / la prediccion
ROJO   = "#B7352D"   # el error
VERDE  = "#8E9C1E"   # los datos reales
PETROL = "#00A0C6"   # la comparacion
GRIS   = "#6F6F6F"   # ejes y contexto

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass
from ipywidgets import interact, FloatSlider, Dropdown, IntSlider

print("Listo.")

---
## 1. Una neurona

Una neurona hace **dos cosas**:

1. Una suma con pesos: $z = w\,x + b$. Eso es una recta.
2. La aplasta con una función curva: $h = \tanh(z)$.

El resultado es siempre la misma forma de "S". Lo único que cambia es
**dónde** está el escalón (`b`) y **qué tan brusco** es (`w`).

In [ ]:
def neurona(x, w, b):
    return np.tanh(w * x + b)

x = np.linspace(-4, 4, 400)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
for w in [0.5, 1, 3, 8]:
    a1.plot(x, neurona(x, w, 0), lw=2, label=f"w={w}")
a1.set_title("w cambia qué tan brusco es el escalón"); a1.legend(fontsize=8)
for b in [-4, -1, 0, 3]:
    a2.plot(x, neurona(x, 2, b), lw=2, label=f"b={b}")
a2.set_title("b cambia dónde está el escalón"); a2.legend(fontsize=8)
for a in (a1, a2):
    a.set_xlabel("x"); a.set_ylabel("h"); a.grid(alpha=.25); a.set_ylim(-1.3, 1.3)
plt.tight_layout(); plt.show()

In [ ]:
# Muevelo tu mismo.
def ver_neurona(w=2.0, b=0.0):
    fig, ax = plt.subplots(figsize=(5.5, 2.8))
    ax.plot(x, neurona(x, w, b), color=AZUL, lw=2.5)
    ax.axvline(-b / w if w != 0 else 0, color=ROJO, ls="--", lw=1)
    ax.set_ylim(-1.3, 1.3); ax.grid(alpha=.25)
    ax.set_xlabel("x"); ax.set_ylabel("h = tanh(w·x + b)")
    ax.set_title(f"el escalón está en x = {-b/w if w!=0 else 0:.2f}")
    plt.show()

interact(ver_neurona,
         w=FloatSlider(value=2.0, min=-8, max=8, step=.25, description="w"),
         b=FloatSlider(value=0.0, min=-8, max=8, step=.25, description="b"))

---
## 2. Sumar neuronas construye cualquier forma (hazlo a mano)

Una neurona sola solo sabe hacer una "S". Si sumamos varias, cada una con su
**amplitud** $c_i$, aparecen formas mucho más ricas:

$$ y = c_1 h_1(x) + c_2 h_2(x) + c_3 h_3(x) + d $$

Abajo hay tres neuronas ya colocadas en tres sitios distintos. **Mueve las
tres amplitudes y la altura `d` hasta que la curva azul tape a la verde.**

Esta curva verde sí se puede alcanzar exactamente con estas tres neuronas.
El error puede llegar a **0**. Inténtalo antes de seguir.

In [ ]:
NEURONAS = [(2.0, 4.0), (3.0, 0.0), (2.5, -5.0)]      # (w, b) de cada neurona

def combinar(x, c1, c2, c3, d):
    return sum(c * neurona(x, w, b) for c, (w, b) in zip([c1, c2, c3], NEURONAS)) + d

# La curva verde esta construida con estas mismas tres neuronas, asi que
# existe una combinacion exacta. (Si te rindes, mira el final de la celda.)
def objetivo_facil(x):
    return combinar(x, 1.5, -1.0, 0.8, 0.2)

def ver_combinacion(c1=0.0, c2=0.0, c3=0.0, d=0.0):
    y = combinar(x, c1, c2, c3, d)
    mse = np.mean((y - objetivo_facil(x)) ** 2)
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.plot(x, objetivo_facil(x), color=VERDE, lw=4, alpha=.6, label="la curva que queremos")
    ax.plot(x, y, color=AZUL, lw=2.2, label="tu suma de 3 neuronas")
    for c, (w, b) in zip([c1, c2, c3], NEURONAS):
        ax.plot(x, c * neurona(x, w, b), color=GRIS, ls="--", lw=1)
    ax.set_ylim(-4, 4); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="upper left")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    marca = "  <- PERFECTO" if mse < 1e-9 else ""
    ax.set_title(f"error (MSE) = {mse:.4f}{marca}")
    plt.show()

interact(ver_combinacion,
         c1=FloatSlider(value=0.0, min=-3, max=3, step=.1, description="c1"),
         c2=FloatSlider(value=0.0, min=-3, max=3, step=.1, description="c2"),
         c3=FloatSlider(value=0.0, min=-3, max=3, step=.1, description="c3"),
         d=FloatSlider(value=0.0, min=-2, max=2, step=.1, description="d"))
# Solucion: c1=1.5, c2=-1.0, c3=0.8, d=0.2

### Tres no siempre alcanzan

Esa curva era fácil porque estaba hecha con esas tres neuronas. Ahora
probemos con una curva de verdad, que es la que usaremos en el resto del
notebook:

$$ f(x) = \sin(2.5\,x) + 0.3\,x $$

No hay que adivinar el mejor ajuste posible: se calcula exacto con mínimos
cuadrados.

In [ ]:
def objetivo(x):
    return np.sin(2.5 * x) + 0.3 * x

# Mejor combinacion posible de estas 3 neuronas para la curva dificil.
A = np.column_stack([neurona(x, w, b) for w, b in NEURONAS] + [np.ones_like(x)])
mejores, *_ = np.linalg.lstsq(A, objetivo(x), rcond=None)
mse_min = float(np.mean((A @ mejores - objetivo(x)) ** 2))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(x, objetivo(x), color=VERDE, lw=4, alpha=.6, label="la curva dificil")
ax.plot(x, A @ mejores, color=AZUL, lw=2.2, label="lo mejor que pueden 3 neuronas")
ax.set_ylim(-4, 4); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="upper left")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"por mucho que muevas las amplitudes, el MSE no baja de {mse_min:.3f}")
plt.show()
print(f"Amplitudes optimas: {np.round(mejores, 3)}")
print(f"MSE minimo con 3 neuronas: {mse_min:.4f}")
print("Hacen falta mas neuronas. Y hay que colocarlas solas, no a mano.")

Lo que acabas de hacer a mano (probar, mirar el error, corregir) es
**exactamente** lo que hace el gradient descent, solo que la
computadora lo hace miles de veces por segundo y con todas las perillas a
la vez, no solo con las amplitudes.

Para eso necesita el gradiente. Vamos por él.

---
## 3. La regla de la cadena, escrita a mano

Tomemos la red más pequeña posible: una entrada, **una** neurona escondida,
una salida. Son las mismas cuatro líneas de las diapositivas, con una sola
neurona en vez de tres:

$$ \underbrace{z = w\,x + b}_{\text{la neurona}} \qquad
   \underbrace{h = \tanh(z)}_{\text{aplastada}} \qquad
   \underbrace{y = c\,h + d}_{\text{la salida}} \qquad
   \underbrace{E = (y-t)^2}_{\text{el error}} $$

Las perillas son $w$, $b$, $c$ y $d$. La entrada $x$ y la respuesta correcta
$t$ vienen de fuera y no se tocan.

Queremos $\partial E/\partial w$. Pero $w$ aparece **solo en la primera
línea**: llega al error dando un rodeo, primero por $z$, luego por $h$, luego
por $y$. Así que multiplicamos la cadena entera:

$$ \frac{\partial E}{\partial w}
   = \underbrace{\frac{\partial E}{\partial y}}_{2(y-t)}
     \cdot \underbrace{\frac{\partial y}{\partial h}}_{c}
     \cdot \underbrace{\frac{\partial h}{\partial z}}_{1-h^2}
     \cdot \underbrace{\frac{\partial z}{\partial w}}_{x} $$

Cada factor es la derivada de **una sola línea**, y ninguna es difícil.
**Backpropagation es esta multiplicación, hecha capa por capa, de atrás hacia
adelante.** El nombre suena difícil; la operación es una multiplicación.

Abajo lo comprobamos: comparamos la cadena con la derivada numérica.

In [ ]:
# Comprobamos la cadena: la derivada por la cadena contra la numerica.
w, b, c, d = 0.8, -0.3, 1.7, 0.4      # las cuatro perillas
x0, t0 = 1.2, 0.5                     # el dato y su respuesta correcta

def error_pequeno(w, b, c, d):
    h = np.tanh(w * x0 + b)
    y = c * h + d
    return (y - t0) ** 2

h = np.tanh(w * x0 + b)
y = c * h + d

# Los cuatro factores, uno por linea, en el mismo orden que las diapositivas.
dE_dy  = 2 * (y - t0)        # de  E = (y - t)^2
dy_dh  = c                   # de  y = c*h + d
dh_dz  = 1 - h ** 2          # de  h = tanh(z)
dz_dw  = x0                  # de  z = w*x + b

por_la_cadena = {
    "w": dE_dy * dy_dh * dh_dz * dz_dw,   # <- la cadena completa
    "b": dE_dy * dy_dh * dh_dz,           # igual, pero dz/db = 1
    "c": dE_dy * h,                       # solo los dos primeros factores
    "d": dE_dy,                           # solo el primero
}

eps = 1e-6
base = [w, b, c, d]
nombres = ["w", "b", "c", "d"]
print(f"{'perilla':8s} {'por la cadena':>15s} {'numerica':>12s}")
for i, n in enumerate(nombres):
    mas, menos = list(base), list(base)
    mas[i] += eps; menos[i] -= eps
    num = (error_pequeno(*mas) - error_pequeno(*menos)) / (2 * eps)
    print(f"{n:8s} {por_la_cadena[n]:15.8f} {num:12.8f}")
print("\nCoinciden. La cadena es correcta.")
print("Fijate en que c y d reaprovechan factores de w y b: eso es lo que")
print("hace backpropagation, calcular de atras hacia adelante y no repetir.")

---
## 4. La red completa, en numpy

Ahora con `n` neuronas en vez de una. Es la misma cadena, escrita con
matrices. Son 25 líneas y no usa ninguna librería de deep learning.

In [ ]:
def crear(n_neuronas, semilla=0):
    """Las cuatro perillas, con los mismos nombres que en las ecuaciones:
    w y b son de las neuronas, c y d son de la salida."""
    r = np.random.default_rng(semilla)
    return {"w": r.normal(0, 1.0, (1, n_neuronas)),
            "b": r.normal(0, 1.5, (1, n_neuronas)),
            "c": r.normal(0, 1 / np.sqrt(n_neuronas), (n_neuronas, 1)),
            "d": np.zeros((1, 1))}

def adelante(p, X):
    """z = X*w + b,  H = tanh(z),  y = H*c + d.  Las mismas cuatro lineas."""
    H = np.tanh(X @ p["w"] + p["b"])
    return H @ p["c"] + p["d"], H

def atras(p, X, Yt):
    """La regla de la cadena, de la salida hacia la entrada."""
    n = len(X)
    Y, H = adelante(p, X)
    dY = (Y - Yt) / n                 # dL/dY
    dH = dY @ p["c"].T               # ...pasa por c
    dZ = dH * (1 - H ** 2)            # ...pasa por la tanh
    return {"w": X.T @ dZ, "b": dZ.sum(0, keepdims=True),
            "c": H.T @ dY, "d": dY.sum(0, keepdims=True)}

def entrenar(n_neuronas, X, Yt, epocas=12000, guardar_cada=None, semilla=0):
    p = crear(n_neuronas, semilla)
    # Con muchas neuronas la salida se mueve mucho por paso, asi que el
    # learning rate baja cuando la red crece.
    lr = min(0.3, 3.0 / n_neuronas)
    fotos = []
    for e in range(epocas):
        g = atras(p, X, Yt)
        for k in p:
            p[k] -= lr * g[k]
        if guardar_cada and e % guardar_cada == 0:
            fotos.append({k: v.copy() for k, v in p.items()})
    return p, fotos

In [ ]:
# Los datos: 18 puntos con ruido, tomados de una curva que la red no conoce.
rng = np.random.default_rng(3)
X  = np.sort(rng.uniform(-3, 3, 18)).reshape(-1, 1)
Yt = objetivo(X) + rng.normal(0, .15, X.shape)
Xd = np.linspace(-3.4, 3.4, 300).reshape(-1, 1)      # rejilla fina para dibujar

p8, fotos = entrenar(8, X, Yt, epocas=12000, guardar_cada=200)
mse = float(np.mean((adelante(p8, X)[0] - Yt) ** 2))
print(f"8 neuronas, 12000 epocas -> MSE = {mse:.5f}  ({len(fotos)} fotos guardadas)")

In [ ]:
# La red aprendiendo, animada. Cada cuadro son 200 pasos de descenso.
fotos_an = fotos[::]                    # 60 cuadros
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=80)
ax.plot(Xd, objetivo(Xd), color=VERDE, lw=2.5, alpha=.5, label="la curva real")
ax.plot(X, Yt, "o", color=VERDE, ms=7, label="los 18 datos")
curva, = ax.plot([], [], color=AZUL, lw=2.5, label="la red")
ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3, 3); ax.grid(alpha=.25)
ax.legend(fontsize=8, loc="upper left"); ax.set_xlabel("x"); ax.set_ylabel("y")
tit = ax.set_title("")

def animar(k):
    Yp, _ = adelante(fotos_an[k], Xd)
    curva.set_data(Xd.ravel(), Yp.ravel())
    e = float(np.mean((adelante(fotos_an[k], X)[0] - Yt) ** 2))
    tit.set_text(f"epoca {k*200:5d}   MSE = {e:.4f}")
    return curva, tit

ani = matplotlib.animation.FuncAnimation(fig, animar, frames=len(fotos_an), interval=90)
plt.close(fig)
ani

---
## 5. La red es un interpolador

Esta es la idea que hay que llevarse del taller.

Una red neuronal **no razona sobre la curva**: la copia por pedazos, con
tantas "S" como neuronas tenga. Es una máquina de interpolar.

Con pocas neuronas no le alcanzan las formas y se queda corta
(*underfitting*). A partir de unas pocas, la curva encaja.

Mediremos dos errores distintos, porque no son lo mismo:

* el error contra **los 18 puntos** que la red vio, y
* el error contra **la curva real**, que la red nunca vio.

El segundo es el que importa.

In [ ]:
TAMANOS = [1, 3, 8, 60]
redes = {n: entrenar(n, X, Yt, epocas=12000)[0] for n in TAMANOS}

def error_datos(p):
    return float(np.mean((adelante(p, X)[0] - Yt) ** 2))

def error_curva_real(p):
    return float(np.mean((adelante(p, Xd)[0] - objetivo(Xd)) ** 2))

fig, axes = plt.subplots(1, 4, figsize=(13, 3.0), sharey=True)
for ax, n in zip(axes, TAMANOS):
    Yp, _ = adelante(redes[n], Xd)
    ax.plot(Xd, objetivo(Xd), color=VERDE, lw=2.5, alpha=.45)
    ax.plot(X, Yt, "o", color=VERDE, ms=6)
    ax.plot(Xd, Yp, color=AZUL, lw=2.2)
    ax.set_ylim(-3, 3); ax.grid(alpha=.25); ax.set_xlabel("x")
    ax.set_title(f"{n} neurona{'s' if n>1 else ''}\n"
                 f"datos {error_datos(redes[n]):.4f}   "
                 f"curva real {error_curva_real(redes[n]):.4f}", fontsize=8.5)
axes[0].set_ylabel("y")
plt.tight_layout(); plt.show()
print("Con el mismo numero de epocas, mas neuronas NO empeora el resultado.")
print("Eso contradice lo que suele contarse. Lo que si empeora es otra cosa.")

### Lo que de verdad produce *overfitting*: entrenar de más

Entrenamos dos redes mucho más tiempo y medimos los dos errores por
separado. El error contra los datos baja siempre. El error contra la curva
real toca un mínimo y después **sube**: a partir de ahí la red está
aprendiendo el ruido de esos 18 puntos.

In [ ]:
def curvas_de_error(n_neuronas, epocas=120000, cada=2000):
    p = crear(n_neuronas)
    lr = min(0.3, 3.0 / n_neuronas)
    pasos, e_datos, e_real = [], [], []
    for e in range(epocas):
        g = atras(p, X, Yt)
        for k in p:
            p[k] -= lr * g[k]
        if e % cada == 0:
            pasos.append(e); e_datos.append(error_datos(p)); e_real.append(error_curva_real(p))
    return np.array(pasos), np.array(e_datos), np.array(e_real)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, n in zip(axes, [8, 60]):
    pasos, e_datos, e_real = curvas_de_error(n)
    ax.plot(pasos, e_datos, color=AZUL, lw=2, label="error contra los 18 datos")
    ax.plot(pasos, e_real, color=ROJO, lw=2, label="error contra la curva real")
    k = int(np.argmin(e_real))
    ax.plot(pasos[k], e_real[k], "*", color=VERDE, ms=16)
    ax.axvline(pasos[k], color=VERDE, ls="--", lw=1)
    ax.set_yscale("log"); ax.set_xlabel("epoca"); ax.grid(alpha=.25)
    ax.set_title(f"{n} neuronas: lo mejor esta en la epoca {pasos[k]:,}", fontsize=9)
axes[0].set_ylabel("MSE (escala log)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("La estrella verde marca cuando hay que parar. Seguir entrenando")
print("mejora los datos que ya viste y empeora todo lo demas.")

In [ ]:
# Elige el tamano y mira TAMBIEN las neuronas por separado (lineas grises).
def ver_red(n_neuronas=8):
    p = redes[n_neuronas]
    Yp, H = adelante(p, Xd)
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.plot(Xd, objetivo(Xd), color=VERDE, lw=2.5, alpha=.45, label="curva real")
    ax.plot(X, Yt, "o", color=VERDE, ms=7, label="datos")
    for j in range(min(n_neuronas, 20)):
        ax.plot(Xd, H[:, [j]] * p["c"][j], color=GRIS, ls="--", lw=.8, alpha=.7)
    ax.plot(Xd, Yp, color=AZUL, lw=2.5, label="la red (suma de todas)")
    ax.set_ylim(-3, 3); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="upper left")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(f"{n_neuronas} neuronas: cada linea gris es UNA neurona")
    plt.show()

interact(ver_red, n_neuronas=Dropdown(options=TAMANOS, value=8,
                                      description="neuronas"))

### El precio: interpola bien, extrapola mal

Dentro del rango de los datos la red acierta. **Fuera**, se va a cualquier
parte, porque no hay nada que la sujete. Una red no sabe nada del mundo:
solo sabe rellenar entre los puntos que vio.

Vale la pena recordarlo cuando un modelo de lenguaje contesta con seguridad
sobre algo que nunca vio.

In [ ]:
Xlejos = np.linspace(-8, 8, 400).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axvspan(-3, 3, color=VERDE, alpha=.10)
ax.text(0, 4.3, "aqui vio datos", ha="center", color=VERDE, fontsize=9)
ax.plot(Xlejos, objetivo(Xlejos), color=VERDE, lw=2.5, alpha=.5, label="curva real")
ax.plot(Xlejos, adelante(redes[60], Xlejos)[0], color=AZUL, lw=2.2, label="la red (60 neuronas)")
ax.plot(X, Yt, "o", color=VERDE, ms=6)
ax.set_ylim(-6, 5); ax.grid(alpha=.25); ax.legend(fontsize=8)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("fuera de la zona verde la red inventa")
plt.show()

---
## 6. Lo mismo en PyTorch

Todo lo anterior en 10 líneas. PyTorch calcula la cadena solo: eso es lo
único que aporta. La idea no cambió.

In [ ]:
import torch, torch.nn as nn

Xt = torch.tensor(X, dtype=torch.float32)
Yy = torch.tensor(Yt, dtype=torch.float32)

red = nn.Sequential(nn.Linear(1, 8), nn.Tanh(), nn.Linear(8, 1))
opt = torch.optim.Adam(red.parameters(), lr=0.05)

for epoca in range(3000):
    perdida_t = ((red(Xt) - Yy) ** 2).mean()
    opt.zero_grad()
    perdida_t.backward()          # <- esta linea ES backpropagation
    opt.step()

print(f"PyTorch, 8 neuronas, 3000 epocas -> MSE = {perdida_t.item():.5f}")

with torch.no_grad():
    Yp_torch = red(torch.tensor(Xd, dtype=torch.float32)).numpy()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(Xd, objetivo(Xd), color=VERDE, lw=2.5, alpha=.45, label="curva real")
ax.plot(X, Yt, "o", color=VERDE, ms=7, label="datos")
ax.plot(Xd, adelante(redes[8], Xd)[0], color=AZUL, lw=2.2, label="nuestra numpy")
ax.plot(Xd, Yp_torch, color=PETROL, lw=2.2, ls="--", label="PyTorch")
ax.set_ylim(-3, 3); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="upper left")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("la misma red, dos veces")
plt.show()

---
## Resumen

1. Una neurona es una recta aplastada. Solo sabe hacer una "S".
2. Sumando muchas "S" se dibuja cualquier curva. Eso es una red.
3. **Backpropagation** es la regla de la cadena aplicada de atrás hacia
   adelante para saber cuánto culpa tiene cada perilla del error.
4. Una red **interpola**. Dentro de los datos acierta; fuera, inventa.
5. Entrenar de más hace *overfitting*: el error contra los datos sigue
   bajando mientras el error de verdad ya subió.

En el siguiente notebook usamos esto para algo que parece muy distinto:
un agente que aprende a moverse solo. Primero con una **tabla**, y cuando
la tabla ya no quepa, con una red.